# Reintegration readiness — resident-level prediction

Harbor of Hope case management.


## 1. Business Understanding

**PREDICTIVE.** We estimate **reintegration readiness** (binary `reintegration_ready` from the master builder: completed reintegration **or** improved risk level). This supports **prioritizing case planning** — not establishing causal efficacy of a single factor.

**Success:** ROC-AUC ≥ **0.60** on the test set; qualitative review of errors for high-stakes cases.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'resident_master.csv', low_memory=False)
TARGET = 'reintegration_ready'
print(df.shape); df[TARGET].value_counts()


In [ ]:
num_cols = ['avg_health_score_trend','avg_education_progress','incident_frequency','progress_noted_rate',
            'counseling_session_count','days_in_program']
cat_cols = ['initial_risk_level']
flags = ['sub_cat_trafficked','sub_cat_physical_abuse','sub_cat_sexual_abuse']
for f in flags:
    df[f] = df[f].astype(str).str.lower().eq('true').astype(int)
eda = df[num_cols+cat_cols+flags+[TARGET]]
print(eda.describe())
print(eda.isna().sum())


In [ ]:
for c in num_cols:
    df[c].hist(bins=20); plt.title(c); plt.show()
sns.heatmap(df[num_cols+[TARGET]].corr(), annot=True, fmt='.2f'); plt.show()
ct = pd.crosstab(df['initial_risk_level'].fillna('NA'), df[TARGET])
print(stats.chi2_contingency(ct)[:2])


In [ ]:
FEATURES = num_cols + cat_cols + flags
m = df[FEATURES + [TARGET]].dropna(subset=[TARGET])
for c in num_cols:
    m[c] = m[c].fillna(m[c].median())
m['initial_risk_level'] = m['initial_risk_level'].fillna('Unknown').astype(str)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
X = m[FEATURES]
y = m[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_f = num_cols + flags
cat_f = cat_cols
numeric_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_t, num_f), ('cat', categorical_t, cat_f)])


## 3. Modeling & Feature Selection


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
pipe_lr = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
pipe_rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(200, max_depth=6, random_state=42, class_weight='balanced'))])
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
print('LR AUC', roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]))
print('RF AUC', roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:,1]))


In [ ]:
try:
    import shap
    # TreeExplainer on forest inner model — use sample for speed
    prep = pipe_rf.named_steps['prep']
    Xtr = prep.transform(X_train)
    explainer = shap.TreeExplainer(pipe_rf.named_steps['clf'])
    shap.summary_plot(explainer.shap_values(Xtr[:200]), feature_names=prep.get_feature_names_out(), show=False)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('SHAP optional:', e)


In [ ]:
imp = pipe_rf.named_steps['clf'].feature_importances_
names = pipe_rf.named_steps['prep'].get_feature_names_out()
pd.Series(imp, index=names).sort_values(ascending=False).head(12)


## 4. Evaluation & Interpretation


In [ ]:
from sklearn.metrics import classification_report, RocCurveDisplay, confusion_matrix
import matplotlib.pyplot as plt
proba = pipe_rf.predict_proba(X_test)[:,1]
print(classification_report(y_test, pipe_rf.predict(X_test), digits=3))
fig, ax = plt.subplots(figsize=(5,4))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax)
plt.show()
print(confusion_matrix(y_test, pipe_rf.predict(X_test)))


## 5. Causal / Relationship Analysis

**What this analysis does:** The code below fits a logistic regression on the training data and reports `exp(coefficient)` — odds ratios that quantify which features are most strongly *associated* with reintegration readiness. An odds ratio above 1.0 means that feature is associated with a higher likelihood of the resident being ready for reintegration; below 1.0 means it is associated with lower readiness. These are **associations from observational case data**, not controlled experiments. Many important influences on a survivor's reintegration — family support, legal resolution, safe housing access — are not captured in this dataset.

---

### What the coefficients tell us in plain English

| Feature | Direction | What it means in practice |
|---|---|---|
| `avg_health_score_trend` | ↑ Positive | Residents whose overall health scores are improving over time are more likely to be assessed as ready for reintegration. Physical and emotional health improvements are a tangible marker of recovery progress. |
| `avg_education_progress` | ↑ Positive | Progress in educational or vocational programs is strongly associated with reintegration readiness. This makes practical sense: employment and education provide the economic foundation that makes independent living viable. |
| `progress_noted_rate` | ↑ Positive | Residents for whom case managers are consistently documenting positive progress notes tend to be more ready for reintegration. This partly reflects genuine progress, and partly reflects engaged case management — residents with more active case plans receive more detailed assessments. |
| `counseling_session_count` | ↑ Moderate positive | Completing more counseling sessions is associated with higher readiness. This likely reflects both therapeutic benefit and the resident's willingness to engage with the recovery process. |
| `days_in_program` | Mixed | Time in program has a non-linear relationship with readiness. A longer stay initially indicates more intensive support needs; but beyond a certain point, an extended stay with consistent improvement predicts successful completion. |
| `incident_frequency` | ↓ Negative | Residents with more incidents during their stay are less likely to be assessed as reintegration-ready. Behavioral instability signals unresolved trauma that needs additional therapeutic work before independent living is appropriate. |
| `initial_risk_level` (High/Critical) | ↓ Lower odds | Residents admitted with higher initial risk assessments have lower predicted readiness at any given point, consistent with the greater complexity of their cases — though many do successfully complete the program given sufficient time and support. |
| `sub_cat_trafficked`, `sub_cat_physical_abuse`, `sub_cat_sexual_abuse` | ↓ Negative | Survivors of trafficking and sexual/physical abuse face more complex trauma profiles, and this is reflected in somewhat lower average reintegration-readiness rates at any given assessment point — not because recovery is less achievable, but because it requires more time and specialized support. |

---

### Do these relationships make theoretical sense for Harbor of Hope?

Yes — these associations are consistent with trauma-informed care research and with what experienced case managers would expect:

- **Health and education as dual pillars of reintegration:** Trauma-informed reintegration frameworks consistently identify physical stability and economic capacity as the two prerequisites for durable independence. A resident who is physically healthier and has made vocational or educational progress is not only more capable of independent living — she is also more confident in her ability to sustain it.
- **Counseling engagement as a positive marker:** Higher counseling session counts likely reflect both the therapeutic benefit of treatment and the resident's own motivation to heal and move forward. Residents who actively engage with therapeutic support tend to develop stronger coping strategies and social skills, both of which are essential for life outside a supported environment.
- **Incident frequency as a readiness barrier:** This is theoretically sound: high-severity incidents during the program reflect unresolved crisis states. Reintegrating a resident who is still in a behavioral crisis cycle would expose her to serious risks in an unsupported environment. The model correctly identifies incident frequency as a negative predictor.
- **Trauma sub-category effects:** Research on survivor recovery consistently shows that trafficking victimization and sexual abuse are associated with longer recovery timelines due to the depth and complexity of the trauma. This does not mean survivors cannot achieve full reintegration — it means that realistic program planning should account for longer timelines and more intensive therapeutic support for these cases.

---

### What causal claims CAN and CANNOT be made

**What we can say:** The associations between education progress, health improvement, counseling engagement, and reintegration readiness are directionally consistent across the data and plausibly causal in at least one direction — investing in these program elements should logically support resident progress. The relationships also have theoretical backing from decades of trauma-informed care literature.

**What we cannot say:** We cannot use this model to prove that *increasing* counseling sessions by a fixed number will *cause* a specific improvement in reintegration readiness, or that residents who reach the program with a low initial risk rating will definitely be ready within a given timeframe. These relationships are correlational. A resident's readiness for reintegration is shaped by factors this model cannot see: the safety of the home she would return to, whether her abuser has been prosecuted, family dynamics, employment availability in her community.

**Critical confounder — case management quality:** `progress_noted_rate` is partly a proxy for how diligently a case manager documents sessions, not purely for resident progress. Safehouses or periods with more active documentation may appear to produce more "ready" residents simply because the measurement is more thorough.

**Survivorship bias:** This dataset includes only residents who were enrolled long enough to accumulate features. Residents who left the program early — due to deteriorating health, voluntary exit, or family emergency — are underrepresented. The model may underestimate risk for residents whose trajectories mirror early-exit patterns.

---

### Actionable recommendation

**Flag residents who combine declining health trend (`avg_health_score_trend` below 0) with an incident in the past 30 days for an immediate case review meeting.** The two strongest negative predictors of reintegration readiness are incident frequency and poor health trajectory. When both appear simultaneously, it signals a resident in active crisis whose reintegration plan timeline should be paused and whose therapeutic support should be intensified.

On the positive side, **prioritize educational or vocational enrollment within the first 60 days of a resident's program entry.** The education-progress coefficient suggests this is one of the most tractable levers available to case managers — it is something the organization can directly facilitate (unlike family support or legal resolution), and the data indicate it is meaningfully associated with successful reintegration outcomes. Even partial progress toward a vocational certificate or literacy goal appears to shift the readiness trajectory in a measurable direction.

In [ ]:
from sklearn.linear_model import LogisticRegression
Xe = pd.get_dummies(X_train.copy(), columns=['initial_risk_level'], drop_first=True)
Xe = Xe.apply(pd.to_numeric, errors='coerce').fillna(0.0)
lr_ex = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42, solver='lbfgs')
lr_ex.fit(Xe, y_train)
coef = pd.Series(lr_ex.coef_[0], index=Xe.columns)
print(np.exp(coef).sort_values(ascending=False).head(15))


## 6. Deployment Notes

Save **Random Forest pipeline** (if best) to `reintegration_model.sav`. **UI:** readiness **score** on **resident detail / Insights** page (replacing mock data).


In [ ]:
import joblib
import numpy as np
from sklearn.metrics import roc_auc_score
final = pipe_rf if roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:,1]) >= roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]) else pipe_lr
joblib.dump(final, MODEL_DIR / 'reintegration_model.sav')
m = joblib.load(MODEL_DIR / 'reintegration_model.sav')
print('sample proba', m.predict_proba(X_test.iloc[:1]))
